In [ ]:
'''
TODO: This file recreates a large number of functionality that has since been
added to fit_utils.py, cut_utils.py, plot_utils.py, fit_dcb.py, fit_gauss.py.
Update the code in this file to use the new functionality.

Basically, the structure now looks like:
1. Prepare the data
2. Pass data to fit_*.py to perform the fit and return the results.
3. Pass the results to plot_mass_fit.py to produce the plots.

We will need to manually plot the dalitz fit.
'''

In [ ]:
# Imports
import sys
import os
import json
import argparse
import numpy as np
import matplotlib.pyplot as plt
import uproot
# Ensure src/utils is importable regardless of working directory.
# Notebooks don't define __file__, so fall back to cwd (VS Code/Jupyter
# run notebooks with cwd set to the notebook's own directory).
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
sys.path.insert(0, os.path.join(_here, '..', 'utils'))
sys.path.insert(0, os.path.join(_here, '..', 'fit'))
sys.path.insert(0, os.path.join(_here, '..', 'plot'))
import fit_utils
import cut_utils
from plot_mass_fit import plot_fit_data
from plot_gauss import build_text
from fit_gauss import fit_gauss

In [ ]:
def invariant_mass2(px1, py1, pz1, e1, px2, py2, pz2, e2):
    """Compute M^2 of the sum of two 4-vectors."""
    e  = e1  + e2
    px = px1 + px2
    py = py1 + py2
    pz = pz1 + pz2
    return e**2 - px**2 - py**2 - pz**2

In [ ]:
# Manually define args, since argparse doesn't work in jupyter notebooks.
infile = '../../ntuple/lhcb/eta2mumugamma/20260608/combined_files.root'
outdir = '../../out/eta2mumugamma/turbo/20260608/'
results_filename = 'dalitz_fit_results.json'
fit_plot_filename = 'dalitz_fit_plot.png'

In [ ]:
def get_arrays(infile, branch_names):
    """Get arrays from branches."""
    # Process ROOT file
    with uproot.open(infile) as f:
        tree = f['tree']
        return tree.arrays(branch_names, library='np')

# Extract branches
branches = get_arrays(infile, ['prt_pid', 'prt_px', 'prt_py', 'prt_pz', 'prt_e', 'tag_dtf_chi2', 'tag_dtf_m'])
# Unzip branches into separate variables
pids = branches['prt_pid']
pxs = branches['prt_px']
pys = branches['prt_py']
pzs = branches['prt_pz']
es = branches['prt_e']
tag_dtf_chi2 = branches['tag_dtf_chi2']
tag_dtf_m = branches['tag_dtf_m']

In [ ]:
print(f'pids: {pids}')
n = 0
for pid in pids:
    if len(pid) > 3: n += 1

print(f'{n} / {len(pids)} have >1 candidate')

In [ ]:
def compute_candidate(pids, pxs, pys, pzs, es, ms, metrics=None):
    """Find the best candidate in one event; return its (m12, m23, mass).
    Selects the candidate with the smallest metric.
    Returns (None, None, None) if no candidate is found.
    """
    valid_daughters, valid_indices = [], []
    # Look over candidates in one event.
    for j in range(len(ms)):
        p1 = p2 = p3 = None  # gamma, mu+, mu-
        # Loop over daughters in one candidate.
        for k in range(3):
            idx = j * 3 + k
            pid = pids[idx]
            px = pxs[idx] / 1000.
            py = pys[idx] / 1000.
            pz = pzs[idx] / 1000.
            e = es[idx] / 1000.
            if pid == 22: p1 = (px, py, pz, e)
            elif pid == -13: p2 = (px, py, pz, e)
            elif pid == 13: p3 = (px, py, pz, e)
        # If all three daughters are found, add to valid candidates.
        if None not in (p1, p2, p3):
            valid_daughters.append((p1, p2, p3))
            valid_indices.append(j)

    # If no valid candidates, return None.
    if not valid_indices: return None, None, None

    # Filter metric values to only include valid candidates.
    valid_metrics = [metrics[k] for k in valid_indices]
    # Find the best candidate based on metrics. This is a plain argmin over a
    # short Python list of 4-vector tuples built inside this event loop;
    # cut_utils.find_best_candidates() applies to whole awkward branches
    # instead, so it doesn't fit here.
    # filtered_idx is the index of the best candidate in valid_daughters, but
    # it could be different from the index in ms, which is why we're not
    # calling it "best_idx" here.
    filtered_idx = int(np.argmin(valid_metrics))
    best_candidate = valid_daughters[filtered_idx]
    # Get the index of the best candidate in the original list of candidates.
    # This is necessary because malformed candidates are filtered out, so the
    # index in valid_daughters may not match the index in ms unless we map it
    # back to the original list.
    best_idx = valid_indices[filtered_idx]

    # Compute invariant masses for the best candidate.
    p1, p2, p3 = best_candidate
    m12 = invariant_mass2(*p1, *p2)   # m^2(gamma, mu+)
    m23 = invariant_mass2(*p2, *p3)   # m^2(mu+, mu-)
    return m12, m23, ms[best_idx]


def compute_dalitz_and_mass(pids, pxs, pys, pzs, es, ms, metrics):
    """Compute (m12, m23, mass) for the best candidate in every event.
    Returns three arrays: m12s, m23s, best_masses.
    """
    m12s, m23s, best_masses = [], [], []
    for i in range(len(pids)):
        # Compute the best candidate for this event.
        m12, m23, mass = compute_candidate(pids[i], pxs[i], pys[i], pzs[i], es[i],
                                            ms[i], metrics[i])
        # Append to list of valid best candidates if results are valid (not
        # None) and positive.
        if None not in (m12, m23, mass) and m12 > 0 and m23 > 0:
            m12s.append(m12)
            m23s.append(m23)
            best_masses.append(mass)
    # Return as numpy arrays for easier plotting and analysis.
    return np.array(m12s), np.array(m23s), np.array(best_masses)

In [ ]:
m12s, m23s, masses = compute_dalitz_and_mass(pids, pxs, pys, pzs, es, 
                                             tag_dtf_m, tag_dtf_chi2)

In [ ]:
# Check ranges and shapes. m12/m23 ranges should match what we saw before
# this refactor (m12: [0.0119, 0.9969], m23: [0.0447, 1.1849]) -- if they
# don't, something about the candidate selection changed behavior.
print(f'--- Dalitz Plot Statistics ---')
print(f'm12 range: [{m12s.min():.4f}, {m12s.max():.4f}]')
print(f'm23 range: [{m23s.min():.4f}, {m23s.max():.4f}]')
# Note only 5 values are <0
print(f'mass range: [{masses.min():.4f}, {masses.max():.4f}]')

# Shape check -- m12s/m23s/masses are now guaranteed the same length by
# construction (compute_dalitz_and_mass appends all three together under
# one condition), so this just confirms that, rather than hoping two
# separately-filtered arrays happened to end up the same length.
print(f'--- Shape check --- ')
print(f'm12s shape: {m12s.shape}')
print(f'm23s shape: {m23s.shape}')
print(f'masses shape: {masses.shape}')

In [ ]:
# Physical constraints for the dalitz plot
M_MU = 0.105658
M_ETA = 0.547862
M12_PHYS = (M_MU**2, (M_ETA - M_MU)**2)
M23_PHYS = ((2 * M_MU)**2, M_ETA**2)

# Test implementation with a quick plot of the Dalitz distribution
plt.figure(figsize=(8, 6))
plt.hist2d(m12s, m23s, bins=100, cmap='viridis', range=[[0, 1.0], [0, 1.2]])
plt.colorbar(label='Counts')

# Mark the physical signal boundary (see M12_PHYS/M23_PHYS above).
plt.axvline(M12_PHYS[0], color='red', ls='--', lw=1, label='physical signal window')
plt.axvline(M12_PHYS[1], color='red', ls='--', lw=1)
plt.axhline(M23_PHYS[0], color='red', ls='--', lw=1)
plt.axhline(M23_PHYS[1], color='red', ls='--', lw=1)

plt.legend(loc='upper right', fontsize=8)
plt.xlabel(r'$m^2_{\gamma,\mu^+}$ [GeV$^2$]')
plt.ylabel(r'$m^2_{\mu^+,\mu^-}$ [GeV$^2$]')

In [ ]:
# Test implementation with a quick plot of the Dalitz distribution in physical range
plt.figure(figsize=(8, 6))
plt.hist2d(m12s, m23s, bins=100, cmap='viridis', range=[[0, 0.22], [0, 0.35]])
plt.colorbar(label='Counts')

# Mark the physical signal boundary (see M12_PHYS/M23_PHYS above).
plt.axvline(M12_PHYS[0], color='red', ls='--', lw=1, label='physical signal window')
plt.axvline(M12_PHYS[1], color='red', ls='--', lw=1)
plt.axhline(M23_PHYS[0], color='red', ls='--', lw=1)
plt.axhline(M23_PHYS[1], color='red', ls='--', lw=1)

plt.legend(loc='upper right', fontsize=8)
plt.xlabel(r'$m^2_{\gamma,\mu^+}$ [GeV$^2$]')
plt.ylabel(r'$m^2_{\mu^+,\mu^-}$ [GeV$^2$]')

In [ ]:
# Perform fit on the mass distribution.
# fit_gauss() takes an array of masses; reading, selection and best-candidate
# reduction happen upstream, which here is compute_dalitz_and_mass() above. So
# this global fit runs on the same sample as the per-bin fits below, and the
# mean/sigma/c0/c1 it produces seed those bins consistently.
result = fit_gauss(masses, xmin=400.0, xmax=696.0, nbins=80,
                   mean_init=547.9, sigma_init=10.0)

In [ ]:
# Plot results of the fit
# plot_fit_data (not plot_fit) since we already have `result` in memory --
# plot_fit is a CLI-only wrapper that parses sys.argv and loads its own JSON
# from disk, which doesn't apply here.
plot_config = {
    'output': os.path.join(outdir, 'global_gauss_fit.png'),
    'title': 'Eta Mass (global fit)',
    'build_text': build_text,
}
plot_fit_data(result, plot_config)

In [ ]:
# Now repeat for per-bin fitting based on m12 and m23
# 1. Define bins
# 2. Loop over bins
# 3. Perform fit in each bin

# The mass fit window and its binning are notebook-level constants because the
# per-bin fit and the per-bin plots (bottom cell) MUST use the same three
# numbers -- otherwise the histogram being reviewed is not the histogram that
# was fit. Keeping them here, next to the fit, is what stops the two from
# drifting apart.
XMIN, XMAX, MASS_NBINS = 400.0, 696.0, 80


def fit_gauss_per_bin(masses, xmin=XMIN, xmax=XMAX, nbins=MASS_NBINS,
                      mean_init=547.9, sigma_init=10.0, c0_init=0.0, c1_init=0.0):
    """Fit a Gaussian to the mass distribution within one Dalitz bin.

    `nbins` is the number of MASS bins across [xmin, xmax] -- it has nothing to
    do with the number of Dalitz bins. The caller must not pass its Dalitz
    binning down into this argument (see fig_gauss_binned).
    """
    # Bin this Dalitz bin's masses across the fit window
    counts, edges = np.histogram(masses, bins=nbins, range=(xmin, xmax))
    counts = counts.astype(float)  # Ensure counts are float for fitting
    N = counts.sum()

    # Fit Gaussian to histogram
    cost = fit_utils.make_gauss_cost(counts, edges, xmin, xmax)

    # Use global fit parameters as initial values for per-bin fits.
    init = dict(
        mean=mean_init,
        sigma=sigma_init,
        n_sig = N * 0.5,
        c0=c0_init,
        c1=c1_init
    )
    limits = dict(
        mean  = (xmin, xmax),
        sigma = (1.0, 50.0),
        n_sig = (0.0, N),
        c0    = (-1.0, 1.0),
        c1    = (-1.4, 1.4),
    )

    # m == fit result object containing fit parameters, errors, and goodness-of-fit metrics
    # Only fix the signal shape parameters for the per-bin fits. The signal may
    # vary across the Dalitz plot somewhat, but we expect the background shape
    # to change to be more significant across the Dalitz plot. n_sig always
    # floats.
    #
    # c0/c1 must NOT be listed here: fixing them pins the background to the
    # single global shape, so every bin's background is the same curve with only
    # its normalization (n_bkg = n_cand - n_sig) allowed to move. That is the
    # opposite of the stated intent above, and it is what makes the background
    # look identical -- and, since the global c1 is small, near-linear -- on
    # every per-bin plot.
    m = fit_utils.run_fit(cost, init, limits, fixed=['mean', 'sigma'])
    return m, N


In [ ]:
def fig_gauss_binned(m12s, m23s, masses, xmin=XMIN, xmax=XMAX, nbins=400,
                     mass_nbins=MASS_NBINS,
                     mean_init=547.9, sigma_init=10.0, c0_init=0.0, c1_init=0.0):
    """Run per-bin Gaussian fit for each bin of m12 and m23. Return a dictionary
    of the fit results and the bin edges for m12 and m23.

    Two independent binnings are in play and they must not be confused:
      nbins      -- total number of DALITZ bins (nbins_per_dim**2)
      mass_nbins -- number of MASS bins each per-bin fit histograms into
    """
    # Define bin edges for m12 and m23
    # Get max and min values for m12 and m23 to define the bin edges OR choose a
    # fixed range for m12 and m23 based on physical considerations
    # m12_min, m12_max = m12s.min(), m12s.max()
    # m23_min, m23_max = m23s.min(), m23s.max()
    m12_min, m12_max = 0.0, 0.21
    m23_min, m23_max = 0.0, 0.31
    # nbins is the total number of bins for the 2D histogram, so we need to take
    # the square root to get the number of bins in each dimension and round to
    # the nearest integer.
    nbins_per_dim = int(np.sqrt(nbins))
    # Now use nbins to define the bin edges for m12 and m23
    m12_edges = np.linspace(m12_min, m12_max, nbins_per_dim + 1)
    m23_edges = np.linspace(m23_min, m23_max, nbins_per_dim + 1)
    # Create a 2D histogram of m12 and m23
    hist2d, m12_edges, m23_edges = np.histogram2d(m12s, m23s, bins=[m12_edges, m23_edges])
    # Prepare a dictionary to hold the fit results for each bin
    fit_results = {}
    # Loop over each bin in the 2D histogram
    for i in range(nbins_per_dim):
        for j in range(nbins_per_dim):
            # Get the bin edges for this bin
            m12_bin_min, m12_bin_max = m12_edges[i], m12_edges[i + 1]
            m23_bin_min, m23_bin_max = m23_edges[j], m23_edges[j + 1]
            # Select masses that fall within this bin
            mask = (
                (m12s >= m12_bin_min) & (m12s < m12_bin_max) &
                (m23s >= m23_bin_min) & (m23s < m23_bin_max)
            )
            masses_in_bin = masses[mask]
            # If there are enough events in the bin, perform the fit.
            # Pass mass_nbins, NOT nbins -- nbins counts Dalitz bins, and
            # feeding it to the mass histogram slices [xmin, xmax] into 400
            # bins of 0.74 MeV, which is what buries each per-bin fit in
            # Poisson noise (~6 counts/bin instead of ~30).
            if len(masses_in_bin) > 10:  # Arbitrary threshold for fitting
                fit_result, n_cand = fit_gauss_per_bin(masses_in_bin, xmin, xmax,
                                               mass_nbins,
                                               mean_init, sigma_init, c0_init, c1_init)
                fit_results[(i, j)] = {
                    'fit': fit_result,
                    'n_cand': n_cand,  # n_cand = n_sig + n_bkg (fixed, not fit)
                    'n_events': len(masses_in_bin),
                    'm12_range': (m12_bin_min, m12_bin_max),
                    'm23_range': (m23_bin_min, m23_bin_max),
                }
    return fit_results, m12_edges, m23_edges

In [ ]:
mean_init = result['fit']['parameters']['mean']['value']
sigma_init = result['fit']['parameters']['sigma']['value']
c0_init = result['fit']['parameters']['c0']['value']
c1_init = result['fit']['parameters']['c1']['value']
# nbins = Dalitz bins (20x20). MASS_NBINS (set with XMIN/XMAX two cells up) is
# the mass binning inside each of those bins -- the two are unrelated.
nbins=400
fit_results, m12_edges, m23_edges = fig_gauss_binned(m12s, m23s, masses, 
                                                     xmin=XMIN, xmax=XMAX, 
                                                     nbins=nbins, 
                                                     mass_nbins=MASS_NBINS,
                                                     mean_init=mean_init, 
                                                     sigma_init=sigma_init, 
                                                     c0_init=c0_init, 
                                                     c1_init=c1_init)
# Extract the number of signal events from the fit results for each bin
signal_counts = np.zeros((int(np.sqrt(nbins)), int(np.sqrt(nbins))))
for (i, j), bin_result in fit_results.items():
    signal_counts[i, j] = bin_result['fit'].values['n_sig'] if bin_result['fit'].valid else 0
# Extract the number of background events from the fit results for each bin.
# n_bkg is not a fit parameter -- it's derived as n_cand - n_sig (unitarity),
# where n_cand was fixed going into the fit (see fig_gauss_binned/fit_gauss_per_bin).
bkg_counts = np.zeros((int(np.sqrt(nbins)), int(np.sqrt(nbins))))
for (i, j), bin_result in fit_results.items():
    bkg_counts[i, j] = (bin_result['n_cand'] - bin_result['fit'].values['n_sig']) if bin_result['fit'].valid else 0


In [ ]:
# Check per-bin fit outcomes
# Hole: signal/backgrounds counts equal 0. Can happen because there are too few
# candidates to attempt a fit (<10) or because the per-bin fit ran but failed to
# converge (m.valid == False). One way to tell the difference is if the hole
# is completely isolated (case a) or surrounded by bins with plenty of
# candidates (case b)
n_per_dim = int(np.sqrt(nbins))
skipped, invalid, valid = [], [], []
for i in range(n_per_dim):
    for j in range(n_per_dim):
        if (i, j) not in fit_results: 
            skipped.append((i, j))
        elif not fit_results[(i, j)]['fit'].valid:
            invalid.append((i, j))
        else:
            valid.append((i, j))

print(f'--- Per-bin fit outcome ({n_per_dim}x{n_per_dim} = {n_per_dim**2} Dalitz bins) ---')
print(f'  skipped (< 10 raw candidates): {len(skipped)}')
print(f'  fit invalid (hole):            {len(invalid)}')
print(f'  fit valid:                     {len(valid)}')

if invalid:
    print(f"\n--- Raw candidate counts at 'invalid fit' holes ---")
    print('(large n_cand means the hole is a fit-convergence failure.')
    for (i, j) in invalid:
        n_cand = fit_results[(i, j)]['n_cand']
        print(f'  bin ({i:2d},{j:2d}): n_cand={n_cand:8.0f}')


In [ ]:
# --- Sanity check B: is the fitted background shape physical? --------------
# c0/c1 now float per Dalitz bin (see the `fixed=` list in fit_gauss_per_bin
# -- currently only "mean" and "sigma" are fixed there). The 2nd-order
# Chebyshev background (cheb_bkg_pdf) is NOT guaranteed non-negative over
# [xmin, xmax] for every (c0, c1) inside its allowed box: e.g. c0 near +/-1
# together with c1 near -/+1.4 pushes the curve below zero at one edge of
# the window. A per-bin fit can land there and still report m.valid == True,
# so this check catches "quietly wrong" bins that a holes-only check (cell
# above) would miss -- these don't show up as 0 in the yield map, they just
# show up as a bogus (but nonzero) yield.
x_check = np.linspace(400.0, 696.0, 200)
neg_bkg_bins = []
for (i, j) in valid:
    r = fit_results[(i, j)]['fit']
    pdf = fit_utils.cheb_bkg_pdf(x_check, 400.0, 696.0, r.values['c0'], r.values['c1'])
    if np.any(pdf < 0):
        neg_bkg_bins.append((i, j))

print(f'--- Background positivity check (valid fits only) ---')
print(f'{len(neg_bkg_bins)} / {len(valid)} valid fits have a background curve '
      f'that goes negative somewhere in [xmin, xmax].')
if neg_bkg_bins:
    print('First few offending bins (i, j, c0, c1):')
    for (i, j) in neg_bkg_bins[:10]:
        r = fit_results[(i, j)]['fit']
        print(f"  ({i:2d},{j:2d}): c0={r.values['c0']:.3f}  c1={r.values['c1']:.3f}")


In [ ]:
# --- Sanity check C: raw candidate density is smooth around each hole -----
# If a "hole" bin's raw n_cand is comparable to its immediate neighbors',
# the 0 written into signal_counts/bkg_counts is purely a fit-convergence
# artifact, not a real gap in the underlying Dalitz distribution.
raw = np.full((n_per_dim, n_per_dim), np.nan)
for (i, j), r in fit_results.items():
    raw[i, j] = r['n_cand']

print(f'--- Neighbor comparison for holes (raw n_cand) ---')
if not invalid:
    print('No holes from non-convergence this run.')
for (i, j) in invalid:
    neighbors = {(di, dj): raw[i + di, j + dj]
                 for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]
                 if 0 <= i + di < n_per_dim and 0 <= j + dj < n_per_dim
                 and not np.isnan(raw[i + di, j + dj])}
    if neighbors:
        neighbor_str = ', '.join(f'{k}: {v:.0f}' for k, v in neighbors.items())
        print(f'  hole ({i:2d},{j:2d}): n_cand={raw[i, j]:.0f}  neighbors: {neighbor_str}')


In [ ]:
# Plot the dalitz plot using the fit results for signal events
# Physical constraints for the dalitz plot
M_MU = 0.105658
M_ETA = 0.547862
M12_PHYS = (M_MU**2, (M_ETA - M_MU)**2)
M23_PHYS = ((2 * M_MU)**2, M_ETA**2)

# Reset plot settings to defaults
plt.rcdefaults()

fig, ax = plt.subplots(figsize=(7, 6))
mesh = ax.pcolormesh(m12_edges, m23_edges, signal_counts.T, cmap='viridis', shading='auto')
fig.colorbar(mesh, ax=ax, label='Number of signal events')

# Mark the physical signal boundary (see M12_PHYS/M23_PHYS above).
ax.axvline(M12_PHYS[0], color='red', ls='--', lw=1, label='physical signal window')
ax.axvline(M12_PHYS[1], color='red', ls='--', lw=1)
ax.axhline(M23_PHYS[0], color='red', ls='--', lw=1)
ax.axhline(M23_PHYS[1], color='red', ls='--', lw=1)
ax.legend(loc='upper right', fontsize=8)

ax.set_xlabel(r'$m^2_{\gamma,\mu^+}$ [GeV$^2$]')
ax.set_ylabel(r'$m^2_{\mu^+,\mu^-}$ [GeV$^2$]')
ax.set_title(r'$\eta \to \mu\mu\gamma$ Dalitz plot (bin-by-bin fitted yield)')

fig.tight_layout()
fig.show()

In [ ]:
# Repeat for the background events
fig, ax = plt.subplots(figsize=(7, 6))
mesh = ax.pcolormesh(m12_edges, m23_edges, bkg_counts.T, cmap='viridis', shading='auto')
fig.colorbar(mesh, ax=ax, label='Number of background events')

# Mark the physical signal boundary (see M12_PHYS/M23_PHYS above).
ax.axvline(M12_PHYS[0], color='red', ls='--', lw=1, label='physical signal window')
ax.axvline(M12_PHYS[1], color='red', ls='--', lw=1)
ax.axhline(M23_PHYS[0], color='red', ls='--', lw=1)
ax.axhline(M23_PHYS[1], color='red', ls='--', lw=1)
ax.legend(loc='upper right', fontsize=8)

ax.set_xlabel(r'$m^2_{\gamma,\mu^+}$ [GeV$^2$]')
ax.set_ylabel(r'$m^2_{\mu^+,\mu^-}$ [GeV$^2$]')
ax.set_title(r'$\eta \to \mu\mu\gamma$ Dalitz plot (bin-by-bin fitted yield)')

fig.tight_layout()
fig.show()

In [ ]:
# Print 1d plot for each bin fit, one page per Dalitz bin, into a single PDF.
#
# Every bin in the grid gets a page -- including the ones fig_gauss_binned
# skipped (< 10 candidates) and the ones whose fit didn't converge. A missing
# fit says nothing about the data, and a page showing the data with no curve on
# it is exactly what's needed to tell "this bin is empty" apart from "this bin
# has data the fit couldn't handle". So the loop runs over the grid itself and
# looks each bin up with .get(), rather than iterating fit_results (which only
# ever contains the bins that were fit).
#
# PdfPages is the file, so it is opened ONCE outside the loop -- constructing it
# per iteration reopens and truncates the same path, leaving a 1-page PDF. The
# `with` block also writes the page tree even if a bin raises partway through.
from matplotlib.backends.backend_pdf import PdfPages

n_per_dim = len(m12_edges) - 1
# Same window and binning the fits used (XMIN/XMAX/MASS_NBINS, defined with
# fit_gauss_per_bin), so each page shows the histogram that was actually fit.
mass_edges = np.linspace(XMIN, XMAX, MASS_NBINS + 1)
mass_centers = 0.5 * (mass_edges[:-1] + mass_edges[1:])
bin_width = (XMAX - XMIN) / MASS_NBINS

pdf_path = os.path.join(outdir, 'per_bin_fits.pdf')
n_pages = n_fitted = 0
with PdfPages(pdf_path) as pdf:
    for i in range(n_per_dim):
        for j in range(n_per_dim):
            m12_bin_min, m12_bin_max = m12_edges[i], m12_edges[i + 1]
            m23_bin_min, m23_bin_max = m23_edges[j], m23_edges[j + 1]
            # Same half-open convention as fig_gauss_binned, so the page shows
            # the same candidates the fit saw.
            mask = ((m12s >= m12_bin_min) & (m12s < m12_bin_max) &
                    (m23s >= m23_bin_min) & (m23s < m23_bin_max))
            counts, _ = np.histogram(masses[mask], bins=mass_edges)
            counts = counts.astype(float)

            # .get() -> None for bins that were never fit at all.
            bin_result = fit_results.get((i, j))
            fit_result = bin_result['fit'] if bin_result is not None else None
            has_fit = fit_result is not None and fit_result.valid

            # Fixed two-panel geometry on every page (spectrum + pulls) so the
            # pages stay comparable when flipping through a few hundred of them.
            fig, (ax, ax_pull) = plt.subplots(
                2, 1, figsize=(8, 6.5), sharex=True,
                gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.07})

            # Data with Poisson error bars -- points, not a filled histogram,
            # because the whole point of these pages is judging how well a curve
            # tracks the data given the per-bin statistics.
            ax.errorbar(mass_centers, counts, yerr=np.sqrt(counts), fmt='o',
                        ms=2.5, lw=1, color='black', label='Data', zorder=3)

            if has_fit:
                p = fit_result.values
                n_sig = p['n_sig']
                n_bkg = bin_result['n_cand'] - n_sig
                # Expected counts per bin from the CDF differences -- the same
                # quantity the ExtendedBinnedNLL compared against, so the drawn
                # curve is the fit rather than a re-derivation of it.
                bin_sig = np.diff(fit_utils.gauss_cdf(mass_edges, p['mean'], p['sigma'],
                                                      XMIN, XMAX)) * n_sig
                bin_bkg = np.diff(fit_utils.cheb_cdf(mass_edges, XMIN, XMAX,
                                                     p['c0'], p['c1'])) * n_bkg
                bin_fit = bin_sig + bin_bkg

                # Smooth curves for display: unit-normalized PDFs scaled by their
                # yield and the bin width to land on the counts scale.
                x_fit = np.linspace(XMIN, XMAX, 400)
                y_sig = n_sig * bin_width * fit_utils.gauss_pdf(
                    x_fit, p['mean'], p['sigma'], xmin=XMIN, xmax=XMAX)
                y_bkg = n_bkg * bin_width * fit_utils.cheb_bkg_pdf(
                    x_fit, XMIN, XMAX, p['c0'], p['c1'])
                ax.plot(x_fit, y_sig + y_bkg, color='red', lw=1.5, label='Fit', zorder=2)
                ax.plot(x_fit, y_bkg, color='green', ls='--', lw=1.2,
                        label='Background', zorder=2)

                # Pearson pulls and chi2, matching fit_gauss's convention.
                ok = bin_fit > 0
                pulls = np.where(ok, (counts - bin_fit) / np.sqrt(np.where(ok, bin_fit, 1)), 0.0)
                chi2 = float(np.sum((counts[ok] - bin_fit[ok])**2 / bin_fit[ok]))
                ndof = int(ok.sum()) - fit_result.nfit
                chi2_ndf = chi2 / ndof if ndof > 0 else float('nan')

                ax_pull.bar(mass_centers, pulls, width=bin_width, color='red',
                            edgecolor='none')
                ax_pull.set_ylim(-5, 5)
                for y in (-2, 2):
                    ax_pull.axhline(y, color='gray', ls=':', lw=0.8)

                status = (f"n_cand = {bin_result['n_cand']:.0f}\n"
                          f"n_sig  = {n_sig:.1f} $\\pm$ {fit_result.errors['n_sig']:.1f}\n"
                          f"n_bkg  = {n_bkg:.1f}\n"
                          f"c0 = {p['c0']:.3f}   c1 = {p['c1']:.3f}\n"
                          f"$\\chi^2$/ndf = {chi2_ndf:.2f}")
                box_color = 'black'
            else:
                # No curve to draw: say why, and leave the pull panel blank
                # rather than faking residuals against a fit that doesn't exist.
                if bin_result is None:
                    status = (f'NO FIT ATTEMPTED\n'
                              f'{int(mask.sum())} candidates (threshold is 10)')
                else:
                    status = (f"FIT DID NOT CONVERGE\n"
                              f"n_cand = {bin_result['n_cand']:.0f}")
                ax_pull.text(0.5, 0.5, 'no fit', transform=ax_pull.transAxes,
                             ha='center', va='center', color='gray', fontsize=9)
                ax_pull.set_ylim(-5, 5)
                box_color = 'red'

            ax.text(0.02, 0.97, status, transform=ax.transAxes, va='top', ha='left',
                    fontsize=8, family='monospace', color=box_color,
                    bbox=dict(boxstyle='round', fc='white', ec=box_color, alpha=0.8))
            ax.set_title(f'Bin ({i}, {j}): '
                         f'm12=[{m12_bin_min:.4f}, {m12_bin_max:.4f}], '
                         f'm23=[{m23_bin_min:.4f}, {m23_bin_max:.4f}]')
            ax.set_ylabel(f'Counts / {bin_width:.2f} MeV')
            ax.set_xlim(XMIN, XMAX)
            ax.legend(loc='upper right', fontsize=8)
            ax_pull.set_xlabel('Mass [MeV]')
            ax_pull.set_ylabel('Pull')

            # One page per bin, then close so a few hundred figures don't all
            # stay resident in memory.
            pdf.savefig(fig)
            plt.close(fig)
            n_pages += 1
            n_fitted += int(has_fit)

print(f'Wrote {n_pages} pages to {pdf_path} '
      f'({n_fitted} with a fit, {n_pages - n_fitted} data-only)')